# Homework 3: Machine Learning for Classification

2026 lead-scoring dataset. The notebook uses the specified missing-value rules,
train/validation split, one-hot encoding, and logistic regression settings.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
local_csv = Path("course_lead_scoring_2026.csv")
df = pd.read_csv(local_csv if local_csv.exists() else DATA_URL)

categorical = df.select_dtypes(include="object").columns.tolist()
numerical = df.select_dtypes(include="number").columns.drop("converted").tolist()
print(f"Rows: {len(df)}")
print("Missing values:", df.isna().sum().to_dict())

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)


Rows: 5000
Missing values: {'lead_source': 148, 'industry': 240, 'employment_status': 194, 'location': 208, 'annual_income': 369, 'number_of_courses_viewed': 0, 'interaction_count': 0, 'lead_score': 35, 'converted': 0}


## Question 1: Most frequent industry

In [2]:
industry_counts = df["industry"].value_counts()
print(industry_counts.to_string())
print("Answer:", industry_counts.idxmax())


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Answer: technology


## Question 2: Largest correlation among the listed pairs

In [3]:
correlation = df[numerical].corr()
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
pair_correlations = {
    pair: correlation.loc[pair[0], pair[1]] for pair in pairs
}
for pair, value in pair_correlations.items():
    print(f"{pair[0]} / {pair[1]}: {value:.6f}")
print("Answer:", max(pair_correlations, key=pair_correlations.get))


interaction_count / lead_score: 0.915746
number_of_courses_viewed / lead_score: 0.757204
number_of_courses_viewed / interaction_count: 0.721609
annual_income / interaction_count: 0.122842
Answer: ('interaction_count', 'lead_score')


## Split the data

In [4]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train["converted"]
y_val = df_val["converted"]
X_train = df_train.drop(columns="converted")
X_val = df_val.drop(columns="converted")
print("Train / validation / test:", len(X_train), len(X_val), len(df_test))


Train / validation / test: 3000 1000 1000


## Question 3: Mutual information on the training set

In [5]:
mi_scores = {
    column: mutual_info_score(X_train[column], y_train)
    for column in categorical
}
for column, score in mi_scores.items():
    print(f"{column}: {score:.6f} (rounded: {round(score, 2):.2f})")
print("Answer:", max(mi_scores, key=mi_scores.get))


lead_source: 0.034346 (rounded: 0.03)
industry: 0.001995 (rounded: 0.00)
employment_status: 0.018966 (rounded: 0.02)
location: 0.001043 (rounded: 0.00)
Answer: lead_source


## Questions 4 and 5: Logistic regression and feature elimination

In [6]:
def validation_accuracy(columns, C=1.0):
    vectorizer = DictVectorizer(sparse=True)
    train_matrix = vectorizer.fit_transform(
        X_train[columns].to_dict(orient="records")
    )
    val_matrix = vectorizer.transform(
        X_val[columns].to_dict(orient="records")
    )
    model = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000, random_state=42
    )
    model.fit(train_matrix, y_train)
    return accuracy_score(y_val, model.predict(val_matrix))


features = X_train.columns.tolist()
baseline = validation_accuracy(features)
print(f"Q4 accuracy: {baseline:.6f} (rounded: {round(baseline, 2):.2f})")

elimination = {
    feature: validation_accuracy([col for col in features if col != feature])
    for feature in features
}
for feature, score in elimination.items():
    print(f"Without {feature}: accuracy={score:.6f}, difference={baseline - score:+.6f}")

q5_options = ["lead_source", "number_of_courses_viewed", "interaction_count"]
print("Q5 answer:", min(q5_options, key=lambda col: baseline - elimination[col]))


Q4 accuracy: 0.645000 (rounded: 0.65)
Without lead_source: accuracy=0.642000, difference=+0.003000
Without industry: accuracy=0.645000, difference=+0.000000
Without employment_status: accuracy=0.645000, difference=+0.000000
Without location: accuracy=0.645000, difference=+0.000000
Without annual_income: accuracy=0.724000, difference=-0.079000
Without number_of_courses_viewed: accuracy=0.643000, difference=+0.002000
Without interaction_count: accuracy=0.601000, difference=+0.044000
Without lead_score: accuracy=0.644000, difference=+0.001000
Q5 answer: number_of_courses_viewed


## Question 6: Regularization strength

In [7]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
regularized = {C: validation_accuracy(features, C=C) for C in c_values}
for C, score in regularized.items():
    print(f"C={C:g}: accuracy={score:.6f}, rounded={round(score, 3):.3f}")
best_C = max(c_values, key=lambda C: (round(regularized[C], 3), -C))
print("Answer:", best_C)


C=1e-06: accuracy=0.598000, rounded=0.598
C=1e-05: accuracy=0.598000, rounded=0.598
C=0.0001: accuracy=0.613000, rounded=0.613
C=0.001: accuracy=0.645000, rounded=0.645
Answer: 0.001


## Answers

| Question | Answer |
| --- | --- |
| 1 | `technology` |
| 2 | `interaction_count` and `lead_score` |
| 3 | `lead_source` |
| 4 | `0.65` |
| 5 | `number_of_courses_viewed` |
| 6 | `0.001` |
